# Week 5 — Protocols, TLS & PKI: secrecy is not authentication

**Lesson plan:** [`../weeks/week-05.md`](../weeks/week-05.md)
**Reading:** the TLS 1.3 handshake overview (RFC 8446 §1–2, skim) + a short
cert-transparency / mis-issuance write-up.

Pure Python, no lab target. Weeks 1–4 secured *messages*. A protocol secures a
*conversation between parties who've never met* — and that turns out to need
something ciphers alone can't give: **authentication**. This week builds the key
exchange, breaks it with a man-in-the-middle, and shows how PKI (certificates)
closes the gap — and how the trust anchor is itself the attack surface.

> ### The one idea
> Diffie–Hellman lets two strangers agree on a secret over a public wire — but it
> proves *secrecy*, not *identity*. A man-in-the-middle defeats unauthenticated DH
> completely. Certificates fix it by **binding a key to a name via a chain of
> signatures up to a trusted root** — so the whole of web security rests on that
> root trust store being correct.

> **Duel 1 (crypto & protocols) is due this week.**

## 1 · Diffie–Hellman — a shared secret over a public channel

Alice and Bob each pick a private number, exchange `g^private mod p` in public, and
each raises the other's public value to their own private power. Both land on
`g^(ab) mod p` — a shared secret an eavesdropper who saw only `g^a` and `g^b` cannot
compute (that's the discrete-log assumption).

In [1]:
import os, hashlib

# A standard 1024-bit MODP group (RFC 2409 group 2). Real, though small by 2025.
p = int("FFFFFFFFFFFFFFFFC90FDAA22168C234C4C6628B80DC1CD129024E08"
        "8A67CC74020BBEA63B139B22514A08798E3404DDEF9519B3CD3A431B"
        "302B0A6DF25F14374FE1356D6D51C245E485B576625E7EC6F44C42E9"
        "A63A3620FFFFFFFFFFFFFFFF", 16)
g = 2

a = int.from_bytes(os.urandom(32), "big")     # Alice's private
b = int.from_bytes(os.urandom(32), "big")     # Bob's private
A, B = pow(g, a, p), pow(g, b, p)             # public, sent over the wire

secret_alice = pow(B, a, p)
secret_bob = pow(A, b, p)
print("Alice and Bob derive the same secret?", secret_alice == secret_bob)
print("an eavesdropper sees only A and B (g^a, g^b) — computing g^(ab) needs a discrete log")

Alice and Bob derive the same secret? True
an eavesdropper sees only A and B (g^a, g^b) — computing g^(ab) needs a discrete log


## 2 · ⚠️ Man-in-the-middle — DH has no idea who it's talking to

DH authenticates *nothing*. If eve sits on the wire, she runs DH with Alice and
a separate DH with Bob, relaying between them. Alice shares a key **with eve**
(thinking it's Bob); Bob likewise. eve decrypts, reads, re-encrypts — invisibly.

In [3]:
m = int.from_bytes(os.urandom(32), "big")     # Eve's private
M = pow(g, m, p)                              # Eve's public, sent to BOTH sides

# Alice sends A toward Bob; Eve intercepts and sends M to each instead.
secret_alice_side = pow(M, a, p)              # Alice thinks this is shared with Bob
secret_eve_a  = pow(A, m, p)                  # Eve's matching key with Alice
secret_bob_side   = pow(M, b, p)
secret_eve_b  = pow(B, m, p)

print("Alice <-> eve share a key:", secret_alice_side == secret_eve_a)
print("Bob   <-> eve share a key:", secret_bob_side == secret_eve_b)
print("Alice <-> Bob share a key    :", secret_alice_side == secret_bob_side, "(NO)")
print("""
Eve holds one key with Alice and another with Bob, and relays every message,
decrypting in the middle. Nobody notices — DH gave perfect SECRECY of each leg and
zero AUTHENTICATION of the endpoints. Secrecy without authentication is a private
conversation with an impostor.""")

Alice <-> eve share a key: True
Bob   <-> eve share a key: True
Alice <-> Bob share a key    : False (NO)

Eve holds one key with Alice and another with Bob, and relays every message,
decrypting in the middle. Nobody notices — DH gave perfect SECRECY of each leg and
zero AUTHENTICATION of the endpoints. Secrecy without authentication is a private
conversation with an impostor.


## 3 · PKI — bind a key to a name with a chain of signatures

The fix: a trusted **Certificate Authority** signs a statement "this public key
belongs to bank.example.com." Your browser ships with the CA's key in a **trust
store**. A certificate chains: the leaf (the website) is signed by an intermediate,
signed by a root the browser trusts. Validation walks the chain to a trusted anchor.

We build a toy signature scheme (RSA, from week 4) and a three-level chain.

In [4]:
import random

def egcd(a, b):
    if b == 0:
        return a, 1, 0
    g, x, y = egcd(b, a % b)
    return g, y, x - (a // b) * y

def modinv(a, m):
    return egcd(a, m)[1] % m

def gen_key(rng, bits=64):
    def prime():
        while True:
            x = rng.getrandbits(bits) | 1 | (1 << (bits - 1))
            if pow(2, x - 1, x) == 1 and all(x % s for s in (3, 5, 7, 11, 13)):
                return x
    pp, qq = prime(), prime()
    n, phi, e = pp * qq, (pp - 1) * (qq - 1), 65537
    return (e, n), (modinv(e, phi), n)         # (public, private)

def digest(msg):
    return int.from_bytes(hashlib.sha256(msg).digest(), "big")

def sign(priv, msg):
    d, n = priv
    return pow(digest(msg) % n, d, n)

def verify(pub, msg, sig):
    e, n = pub
    return pow(sig, e, n) == digest(msg) % n

rng = random.Random(7)
root_pub,  root_priv  = gen_key(rng)
inter_pub, inter_priv = gen_key(rng)
leaf_pub,  leaf_priv  = gen_key(rng)

def make_cert(subject, pub, issuer_priv):
    body = f"{subject}|{pub}".encode()
    return {"subject": subject, "pub": pub, "body": body,
            "sig": sign(issuer_priv, body)}

root_cert  = make_cert("ACME Root CA",      root_pub,  root_priv)   # self-signed
inter_cert = make_cert("ACME Intermediate", inter_pub, root_priv)   # signed by root
leaf_cert  = make_cert("bank.example.com",  leaf_pub,  inter_priv)  # signed by intermediate

TRUST_STORE = {root_pub}          # what the browser ships with
print("built chain: bank.example.com <- ACME Intermediate <- ACME Root CA (trusted)")

built chain: bank.example.com <- ACME Intermediate <- ACME Root CA (trusted)


In [5]:
def validate(chain, trust_store):
    """chain = [leaf, intermediate, ..., root]; each signed by the next."""
    for i in range(len(chain) - 1):
        issuer = chain[i + 1]
        if not verify(issuer["pub"], chain[i]["body"], chain[i]["sig"]):
            return False, f"{chain[i]['subject']}: signature not valid under its issuer"
    root = chain[-1]
    if root["pub"] not in trust_store:
        return False, f"anchor {root['subject']!r} is not a trusted root"
    if not verify(root["pub"], root["body"], root["sig"]):
        return False, "root self-signature invalid"
    return True, "chain valid — key is authentically bound to the name"

ok, reason = validate([leaf_cert, inter_cert, root_cert], TRUST_STORE)
print("legitimate chain:", ok, "-", reason)

legitimate chain: True - chain valid — key is authentically bound to the name


## 4 · ⚠️ Why Eve can't just forge a certificate

Eve can *make* a certificate saying "bank.example.com → my key." What she can't
do is get it signed by a key in your trust store. Two forgery attempts, both
rejected at the trust anchor:

In [ ]:
# Attempt 1: self-signed cert for the bank, using eve's own key.
mal_pub, mal_priv = gen_key(rng)
forged = make_cert("bank.example.com", mal_pub, mal_priv)
print("self-signed forgery :", validate([forged], TRUST_STORE))

# Attempt 2: Eve stands up her own "CA" and signs a bank cert with it.
rogue_ca_pub, rogue_ca_priv = gen_key(rng)
rogue_ca   = make_cert("Rogue CA", rogue_ca_pub, rogue_ca_priv)     # self-signed
rogue_leaf = make_cert("bank.example.com", mal_pub, rogue_ca_priv)  # signed by rogue
print("rogue-CA chain      :", validate([rogue_leaf, rogue_ca], TRUST_STORE))

print("""
Both fail at the SAME check: the chain doesn't terminate in a trusted root. Eve
can sign anything; she cannot make your browser trust her signing key. THAT is what
PKI buys — and it relocates the whole problem to one question: is the trust store
correct?""")

self-signed forgery : (False, "anchor 'bank.example.com' is not a trusted root")
rogue-CA chain      : (False, "anchor 'Rogue CA' is not a trusted root")

Both fail at the SAME check: the chain doesn't terminate in a trusted root. Mallory
can sign anything; she cannot make your browser trust her signing key. THAT is what
PKI buys — and it relocates the whole problem to one question: is the trust store
correct?


### ...which is exactly where real PKI breaks

The math above is sound. Real-world PKI failures are failures of the *trust anchor*,
not the signatures:

- A CA is **compromised or coerced** and mis-issues a valid cert for your domain
  (DigiNotar, 2011 — used to MITM Gmail in Iran).
- A CA is simply **negligent** and signs a cert it shouldn't.
- Malware **installs a rogue root** in your trust store — now the rogue-CA chain
  above validates.

The defenses are themselves protocols: **Certificate Transparency** (all certs
logged publicly, so mis-issuance is detectable), pinning, short-lived certs. The
guarantee (axis 2) is only as strong as the trust store and the CAs in it.

## 5 · TLS 1.3 in one paragraph

Modern TLS composes exactly these pieces: an (authenticated) **Diffie–Hellman** key
exchange for forward-secret session keys, a **certificate chain** to authenticate
the server (§3–4), and **AEAD** (AES-GCM — week 3, nonce discipline and all) for the
bulk data. TLS 1.3 stripped the insecure options TLS 1.2 still allowed (static RSA
key exchange, CBC modes, renegotiation) — a rare case of a protocol getting *simpler
and safer*. Every weakness in this course lives somewhere in that stack: nonce reuse
in the AEAD (wk3), a weak key (wk4), a MITM on unauthenticated DH (§2), a bad trust
anchor (§4).

## 6 · The scorecard view

| Mechanism | Guarantee (axis 2) | Its condition / failure |
|---|---|---|
| Diffie–Hellman | shared secret vs. a passive eavesdropper | **no authentication** — MITM defeats it |
| Certificate chain | binds a key to a name | only as trustworthy as the **root trust store** |
| PKI overall | server authentication | CA compromise / mis-issuance / rogue root |
| TLS 1.3 | confidential, authenticated channel | every underlying condition (nonce, key, trust) must hold |

> Secrecy and authentication are **different properties**, and a protocol needs
> both. DH gives the first; PKI gives the second; TLS composes them. The failures
> aren't in the cryptography — they're in the *conditions*: an unauthenticated
> exchange, a mis-issued cert, a poisoned trust store. Same lesson as every crypto
> week: name the condition, find the attack.

## 7 · Your studio deliverable — and Duel 1

In `week05/`:

1. **DH + MITM** — implement the exchange; mount the man-in-the-middle and show
   Eve holds a key with each side. State what DH does and does not guarantee.
2. **Cert-chain validation** — build a 3-level chain and a validator; then defeat
   two forgeries (self-signed, rogue-CA) and explain why both fail at the anchor.
3. **Trust-store attack** — add the rogue root to the trust store and show the rogue
   chain now validates. This is the real-world failure mode; connect it to
   DigiNotar and to malware-installed roots.
4. **Control Scorecard** — DH, cert chains, and TLS: guarantee + condition for each.

> **Duel 1 (crypto & protocols) is due this week.** It folds in the SECS
> non-repudiation design problem from the old HW2; the DH/PKI material here is
> exactly the toolkit for arguing a protocol's guarantees rigorously.